# SanskritEnv · GRPO on Qwen2.5-1.5B-Instruct (A100, 1 epoch)

End-to-end Colab notebook that:
1. Uses the **deployed** SanskritEnv on Hugging Face (`https://adityahars-sanskrit-env.hf.space`) for `/reset` and `/step` (no local `uvicorn` on this machine).
2. Captures **baseline metrics** from the untrained Qwen 1.5B base model.
3. Trains the policy with **GRPO** for **1 epoch** at **1500 episodes/task** on an A100.
4. Captures **post-training metrics** with the trained LoRA adapter.
5. Renders a side-by-side comparison table showing **per-task improvements** (mean / std / success-rate / full-credit-rate / absolute delta / relative %).

## Why these defaults
| Knob | Value | Reason |
|---|---|---|
| Hardware | A100 80GB | 5–10× the throughput of a T4. Good default for 1-epoch + 1500 ep/task. |
| `EPISODES_PER_TASK` | 1500 | Matches your spec. 1500 × 6 tasks = 9000 unique prompts/epoch. |
| `EPOCHS` | 1.0 | One full pass over the prompt set. Increase to 2.0+ if you have time and need more learning signal. |
| `GROUP_SIZE` | 8 | GRPO’s K. Larger group → better advantage normalization. |
| `PER_DEVICE_BATCH` | 2 | Effective batch = 2 × 8 grad-accum × 8 generations = 128 trajectories per update. |
| `LR` | 5e-6 | LoRA-safe learning rate. |
| 4-bit QLoRA | OFF | A100 has plenty of VRAM; bf16 weights give cleaner gradients. |
| `MAX_COMPLETION_LENGTH` | 96 | SanskritEnv answers are short. |

## Time and cost estimate (A100 80GB on HF, ~$4.13/hr)
| Phase | Episodes | Wall time | Cost |
|---|---:|---:|---:|
| Baseline eval | 6 × 30 = 180 | ~6 min | ~$0.40 |
| Training (1 ep) | 9000 | ~1.0–1.5 h (GPU) | ~$4–6 |
| Post eval | 180 | ~6 min | ~$0.40 |
| **Total** | | **~2 h+** (incl. remote env collection) | **~$6–9** |

## 1. Verify the GPU is an A100

If `nvidia-smi` does NOT show "A100", switch the runtime: Runtime → Change runtime type → GPU → A100. On Colab Pro / Pro+ the A100 is in the dropdown. On HF Spaces / HF Jobs select `a100-large` hardware.

In [ ]:
!nvidia-smi

## 2. Install dependencies

We install **PyTorch + TRL/PEFT** for training only. The env runs on the HF Space (HTTPS). No local FastAPI/uvicorn. Takes ~2–3 min on a fresh runtime.

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet \
    'torch==2.4.1' \
    --index-url https://download.pytorch.org/whl/cu121
%pip install --quiet \
    'transformers>=4.45.0' \
    'accelerate>=0.34.0' \
    'peft>=0.13.0' \
    'trl>=0.13.0' \
    'datasets>=3.0.0' \
    'bitsandbytes>=0.43.0' \
    'sentencepiece' \
    'huggingface-hub>=0.23.0' \
    'requests' \
    'matplotlib>=3.8.0'

## 3. Authenticate with HuggingFace

Add a HuggingFace token under Colab **Secrets** with name `HF_TOKEN` (read+write if you will push the adapter; read-only is enough for model download if Qwen is public). **Never** paste a token in notebook source or commit it. If a token is ever exposed, revoke it in HuggingFace settings and create a new one.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata  # type: ignore
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    if not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = getpass('Paste HF_TOKEN: ')

if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is empty. Add it to Colab Secrets or paste when prompted.')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('Logged into HuggingFace.')

## 4. Clone SanskritEnv

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/aditya-raj9125/Sanskrit-Env.git'
REPO_DIR = '/content/sanskrit-env'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

## 5. Point Colab at the **deployed** HF Space (env URL)

Training uses **this URL** for all env HTTP calls. Override with Colab secret `ENV_URL` if you fork the Space. The Space may be cold: first contact can take 30–120s.

In [ ]:
import time, os, requests

try:
    from google.colab import userdata  # type: ignore
    _u = (userdata.get('ENV_URL') or '').strip()
except Exception:
    _u = os.environ.get('ENV_URL', '').strip()
ENV_URL = _u or 'https://adityahars-sanskrit-env.hf.space'
ENV_URL = ENV_URL.rstrip('/')
os.environ['ENV_URL'] = ENV_URL
os.environ['HF_SPACE_URL'] = ENV_URL
print('ENV_URL =', ENV_URL)

for attempt in range(120):
    try:
        r = requests.get(f'{ENV_URL}/health', timeout=30)
        if r.status_code == 200:
            print('Remote env responded OK after', attempt + 1, 'tries:', r.text[:200])
            break
    except Exception as e:
        if attempt % 20 == 0:
            print('Waiting for cold Space / network...', e)
    time.sleep(2)
else:
    raise RuntimeError('HF Space /health did not return 200. Open the Space in a browser, wait for build, then re-run.')

## 6. Baseline evaluation (BEFORE training)

We evaluate the **untrained Qwen 1.5B base model** so we have a reference point for measuring improvement. 30 episodes per task = 180 total. ~6 min on A100.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
EVAL_EPISODES = 30
EVAL_SEED = 10000
BASELINE_JSON = '/content/runs/eval_baseline.json'
POST_JSON = '/content/runs/eval_post.json'

!python training/evaluate.py \
    --env-url {ENV_URL} \
    --base-model {MODEL_ID} \
    --episodes-per-task {EVAL_EPISODES} \
    --base-seed {EVAL_SEED} \
    --output {BASELINE_JSON} \
    --label baseline-untrained

## 7. Configure GRPO training

These are the A100-tuned defaults. Adjust only if you understand the trade-offs.

In [ ]:
EPISODES_PER_TASK = 1500
GROUP_SIZE = 8
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 4
EPOCHS = 1.0
LR = 5e-6
MAX_COMPLETION_LENGTH = 96
OUTPUT_DIR = '/content/runs/qwen25-1p5b-grpo'
DATASET_CACHE = '/content/runs/prompts.jsonl'
EVAL_EPISODES_DURING_TRAINING = 20  # used by the per-epoch callback inside train_grpo.py
PUSH_TO_HUB = False
HUB_MODEL_ID = None  # e.g. 'your-username/sanskrit-qwen25-1p5b-grpo'

print('Will train', MODEL_ID, 'on', EPISODES_PER_TASK * 6, 'prompts ×', EPOCHS, 'epoch(s).')
print('Output:', OUTPUT_DIR)

## 8. (Optional) Pre-collect the prompt dataset

Caching prompts as JSONL means restarts skip the env-rollout step. ~5 min for 9000 prompts.

In [ ]:
!python training/train_grpo.py \
    --env-url {ENV_URL} \
    --model-id {MODEL_ID} \
    --episodes-per-task {EPISODES_PER_TASK} \
    --base-seed 42 \
    --dataset-cache {DATASET_CACHE} \
    --dry-run

## 9. Train

The trainer:
- Captures a baseline pass before the first step (`--baseline-eval`).
- Runs an eval pass at the end of every epoch (`EpochEvalCallback`).
- Persists `metrics_history.json` inside `OUTPUT_DIR` after every checkpoint, so you can plot the reward curve even if the runtime dies.

Watch the logs for lines like (1 epoch: one `post_epoch` at end, plus `final` if enabled):
```
[eval] epoch=1.00 overall_mean=0.651 std=0.28 success_rate=0.76 (295.0s)
```

In [ ]:
import shlex

cmd = (
    'python training/train_grpo.py'
    f' --env-url {ENV_URL}'
    f' --model-id {MODEL_ID}'
    f' --episodes-per-task {EPISODES_PER_TASK}'
    f' --base-seed 42'
    f' --dataset-cache {DATASET_CACHE}'
    f' --output-dir {OUTPUT_DIR}'
    f' --group-size {GROUP_SIZE}'
    f' --per-device-batch {PER_DEVICE_BATCH}'
    f' --grad-accum {GRAD_ACCUM}'
    f' --epochs {EPOCHS}'
    f' --lr {LR}'
    f' --max-completion-length {MAX_COMPLETION_LENGTH}'
    f' --eval-episodes-per-task {EVAL_EPISODES_DURING_TRAINING}'
    f' --eval-base-seed 10000'
    f' --baseline-eval'
)
if PUSH_TO_HUB and HUB_MODEL_ID:
    cmd += f' --push-to-hub --hub-model-id {shlex.quote(HUB_MODEL_ID)}'

print('Running:\n', cmd)
!{cmd}

## 10. Post-training evaluation (AFTER training)

Loads the trained LoRA adapter on top of the base model and re-runs the same `30 ep/task` eval. Same seeds as baseline, so the comparison is apples-to-apples.

In [ ]:
!python training/evaluate.py \
    --env-url {ENV_URL} \
    --base-model {MODEL_ID} \
    --adapter {OUTPUT_DIR} \
    --episodes-per-task {EVAL_EPISODES} \
    --base-seed {EVAL_SEED} \
    --output {POST_JSON} \
    --label post-train-2ep

## 11. Comparison table

Renders the per-task improvement table with absolute delta and relative %. Also writes a Markdown copy you can drop into your README.

In [ ]:
!python training/compare_evals.py \
    {BASELINE_JSON} \
    {POST_JSON} \
    --markdown /content/runs/improvement_table.md

print()
print('Markdown table:')
with open('/content/runs/improvement_table.md', 'r') as f:
    print(f.read())

## 12. (Optional) Plot the per-epoch reward curve

`metrics_history.json` written by the trainer holds: `baseline`, `post_epoch` after each epoch, and `final`. We chart overall_mean and per-task means so you can see learning dynamics.

In [ ]:
import json, matplotlib.pyplot as plt

history_path = f'{OUTPUT_DIR}/metrics_history.json'
with open(history_path, 'r') as f:
    history = json.load(f)['history']

epochs = [entry['epoch'] for entry in history]
overall = [entry['metrics']['overall']['score_mean'] for entry in history]
labels = [entry['phase'] for entry in history]

plt.figure(figsize=(8, 4))
plt.plot(epochs, overall, marker='o', linewidth=2)
for x, y, lbl in zip(epochs, overall, labels):
    plt.annotate(lbl, (x, y), textcoords='offset points', xytext=(5, 5), fontsize=8)
plt.xlabel('Epoch')
plt.ylabel('Overall mean reward')
plt.title('GRPO training curve')
plt.grid(True, alpha=0.3)
plt.show()

tasks = sorted(history[-1]['metrics']['tasks'].keys())
plt.figure(figsize=(8, 5))
for task in tasks:
    ys = [entry['metrics']['tasks'].get(task, {}).get('score_mean', 0.0) for entry in history]
    plt.plot(epochs, ys, marker='o', label=task)
plt.xlabel('Epoch')
plt.ylabel('Task mean reward')
plt.title('Per-task reward over training')
plt.legend(fontsize=8, loc='best')
plt.grid(True, alpha=0.3)
plt.show()

## 13. (Optional) Push the adapter to the Hub

In [ ]:
from huggingface_hub import HfApi

REPO_TO_PUSH = 'your-username/sanskrit-qwen25-1p5b-grpo'  # change me
api = HfApi()
api.create_repo(REPO_TO_PUSH, repo_type='model', exist_ok=True)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_TO_PUSH, repo_type='model')
print('Uploaded ->', REPO_TO_PUSH)

## 14. GPU session and billing

There is **no** local env server in this notebook. **Colab** stops charging the GPU when you **disconnect the runtime** or the session times out. The **HF Space** may keep running; manage it under [Space settings](https://huggingface.co/spaces/Adityahars/Sanskrit-env) (sleep / hardware) if you pay for a GPU there. For a full runbook on HF account credits, Spaces, and Jobs, read `training/HF_CREDITS_TRAINING.md` in the repo.

In [ ]:
print('Done. Runtime → Disconnect and delete runtime (Colab) to release GPU.')
print('Remote env URL was:', __import__('os').environ.get('ENV_URL'))